# Live Tomo Diff Notebook

This notebook wraps `live_tomo_diff.py` with widgets so you can:

- paste the path to your reference dataset directly
- optionally paste a comparison dataset path, or leave it empty for auto-follow
- pick a projection index across all detector block files
- choose whether live mode should follow only the same position or all positions
- preview a diff image inside the notebook
- launch the live viewer with the selected options

If you leave the comparison path empty, the launched viewer will watch for the newest matching dataset in the collection.

In [ ]:
from pathlib import Path
from io import BytesIO
import shlex
import threading
import time
import sys

import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

import live_tomo_diff as ltd

plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
AUTO_FOLLOW = 'AUTO_FOLLOW'

reference_path_input = widgets.Text(
    value='011_test/011_test_first_position',
    description='Reference path:',
    placeholder='Paste a dataset directory, dataset .h5, or scan .h5 path',
    layout=widgets.Layout(width='900px'),
)
comparison_path_input = widgets.Text(
    value='',
    description='Comparison path:',
    placeholder='Optional. Leave empty for auto-follow latest tomo',
    layout=widgets.Layout(width='900px'),
)
projection_index = widgets.BoundedIntText(
    value=0,
    min=0,
    description='Projection:',
    layout=widgets.Layout(width='250px'),
)
position_mode = widgets.Dropdown(
    options=['same', 'all'],
    value='same',
    description='Positions:',
    layout=widgets.Layout(width='250px'),
)
dataset_path = widgets.Text(
    value='',
    placeholder='Optional exact HDF5 image stack path',
    description='Dataset path:',
    layout=widgets.Layout(width='900px'),
)
poll_interval = widgets.FloatSlider(
    value=2.0,
    min=0.2,
    max=10.0,
    step=0.2,
    description='Poll (s):',
    readout_format='.1f',
    layout=widgets.Layout(width='350px'),
)

collection_dir = widgets.Text(
    value='011_test',
    description='Browse root:',
    placeholder='Optional helper: folder containing dataset directories',
    layout=widgets.Layout(width='900px'),
)
dataset_picker = widgets.Dropdown(
    options=[],
    description='Found datasets:',
    layout=widgets.Layout(width='900px'),
)
refresh_button = widgets.Button(description='Browse datasets', button_style='info')
use_as_reference_button = widgets.Button(description='Use as reference')
use_as_comparison_button = widgets.Button(description='Use as comparison')
preview_button = widgets.Button(description='Preview diff', button_style='success')
command_button = widgets.Button(description='Show command')
start_live_button = widgets.Button(description='Start live viewer', button_style='warning')
stop_live_button = widgets.Button(description='Stop live viewer')
output = widgets.Output()
live_output = widgets.Output()
live_status = widgets.HTML('<b>Live status:</b> idle')
live_image = widgets.Image(format='png', layout=widgets.Layout(width='1100px'))

live_state = {
    'thread': None,
    'stop_event': None,
}


def dataset_choices(collection_path: Path):
    if not collection_path.exists():
        return []
    return [
        str(path.resolve())
        for path in sorted(collection_path.iterdir())
        if ltd.is_dataset_directory(path)
    ]


def refresh_datasets(*_):
    with output:
        clear_output(wait=True)
        collection_path = Path(collection_dir.value).expanduser()
        choices = dataset_choices(collection_path)
        dataset_picker.options = choices
        if choices:
            dataset_picker.value = choices[0]
            print(f'Loaded {len(choices)} dataset directories from {collection_path}')
        else:
            print(f'No dataset directories found in {collection_path}')


def use_selected_as_reference(*_):
    if dataset_picker.value:
        reference_path_input.value = str(dataset_picker.value)


def use_selected_as_comparison(*_):
    if dataset_picker.value:
        comparison_path_input.value = str(dataset_picker.value)


def selected_paths():
    reference_raw = reference_path_input.value.strip()
    comparison_raw = comparison_path_input.value.strip()
    if not reference_raw:
        raise ValueError('Reference path is required.')

    reference_path = Path(reference_raw).expanduser().resolve()
    comparison_path = None if not comparison_raw else Path(comparison_raw).expanduser().resolve()
    explicit_dataset_path = dataset_path.value.strip() or None
    return reference_path, comparison_path, explicit_dataset_path


def resolve_selection():
    reference_path, comparison_path, explicit_dataset_path = selected_paths()
    reference_root, reference_scan = ltd.resolve_input_target(reference_path)
    comparison_root, comparison_scan, auto_follow = ltd.resolve_second_target(
        reference_root,
        comparison_path,
        position_mode.value,
    )
    return reference_root, reference_scan, comparison_root, comparison_scan, auto_follow, explicit_dataset_path


def render_diff_png(diff, title):
    fig = Figure(figsize=(12, 6), dpi=120)
    FigureCanvasAgg(fig)
    ax = fig.add_subplot(111)
    image = ax.imshow(diff, cmap='gray')
    ax.set_title(title)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    fig.colorbar(image, ax=ax, label='Difference')
    fig.tight_layout()

    buffer = BytesIO()
    fig.savefig(buffer, format='png', bbox_inches='tight')
    plt.close(fig)
    return buffer.getvalue()


def preview_diff(*_):
    with output:
        clear_output(wait=True)
        try:
            reference_root, reference_scan, comparison_root, comparison_scan, auto_follow, explicit_dataset_path = resolve_selection()
            ref_count = ltd.scan_projection_count(reference_scan, explicit_dataset_path)
            cmp_count = ltd.scan_projection_count(comparison_scan, explicit_dataset_path)
            idx = projection_index.value
            ref_image = ltd.load_projection_radiogram(reference_scan, idx, explicit_dataset_path)
            cmp_image = ltd.load_projection_radiogram(comparison_scan, idx, explicit_dataset_path)
            diff = cmp_image - ref_image

            print('Reference dataset:', reference_root)
            print('Comparison dataset:', comparison_root)
            print('Reference scan:', reference_scan)
            print('Comparison scan:', comparison_scan)
            print('Auto follow:', auto_follow)
            print('Projection index:', idx)
            print('Reference projections:', ref_count)
            print('Comparison projections:', cmp_count)
            print('Diff min/max/mean:', float(diff.min()), float(diff.max()), float(diff.mean()))

            fig, ax = plt.subplots()
            image = ax.imshow(diff, cmap='gray')
            ax.set_title(f'{comparison_root.name} - {reference_root.name}')
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            plt.colorbar(image, ax=ax, label='Difference')
            plt.show()
        except Exception as exc:
            print(f'Preview failed: {exc}')


def build_command():
    reference_path, comparison_path, explicit_dataset_path = selected_paths()
    command = [sys.executable, 'live_tomo_diff.py', '--reference-path', str(reference_path)]
    if comparison_path is not None:
        command.extend(['--comparison-path', str(comparison_path)])
    command.extend(['--projection-index', str(projection_index.value)])
    command.extend(['--position-mode', position_mode.value])
    command.extend(['--poll-interval', str(poll_interval.value)])
    if explicit_dataset_path is not None:
        command.extend(['--dataset-path', explicit_dataset_path])
    return command


def show_command(*_):
    with output:
        clear_output(wait=True)
        print(' '.join(shlex.quote(part) for part in build_command()))


def stop_live_viewer(*_):
    stop_event = live_state.get('stop_event')
    if stop_event is not None:
        stop_event.set()
    live_status.value = '<b>Live status:</b> stopping...'


def live_worker(reference_root, reference_scan, comparison_root, comparison_scan, auto_follow, explicit_dataset_path, stop_event):
    idx = projection_index.value
    reference_image = ltd.load_projection_radiogram(reference_scan, idx, explicit_dataset_path)
    last_seen_dataset = comparison_root

    while not stop_event.is_set():
        current_root = comparison_root
        current_scan = comparison_scan
        if auto_follow:
            position_name = None
            if position_mode.value == 'same':
                position_name = ltd.dataset_position_name(reference_root, reference_root.parent)
            newest_dataset = ltd.latest_projection_dataset(
                reference_root.parent,
                exclude=reference_root,
                position_name=position_name,
            )
            if newest_dataset is not None:
                current_root = newest_dataset
                current_scan = ltd.find_projection_scan(current_root)

        comparison_image = ltd.load_projection_radiogram(current_scan, idx, explicit_dataset_path)
        diff = comparison_image - reference_image
        live_image.value = render_diff_png(diff, f'{current_root.name} - {reference_root.name} | projection {idx}')
        live_status.value = (
            '<b>Live status:</b> running | '
            f'comparison={current_root.name} | '
            f'projection={idx} | '
            f'updated={time.strftime("%H:%M:%S")}'
        )
        last_seen_dataset = current_root

        if stop_event.wait(poll_interval.value):
            break

    live_status.value = '<b>Live status:</b> stopped'


def start_live_viewer(*_):
    stop_live_viewer()
    with live_output:
        clear_output(wait=True)
        display(live_status)
        display(live_image)
    try:
        reference_root, reference_scan, comparison_root, comparison_scan, auto_follow, explicit_dataset_path = resolve_selection()
        stop_event = threading.Event()
        worker = threading.Thread(
            target=live_worker,
            args=(reference_root, reference_scan, comparison_root, comparison_scan, auto_follow, explicit_dataset_path, stop_event),
            daemon=True,
        )
        live_state['stop_event'] = stop_event
        live_state['thread'] = worker
        live_status.value = '<b>Live status:</b> starting...'
        worker.start()
    except Exception as exc:
        live_status.value = f'<b>Live status:</b> failed: {exc}'


refresh_button.on_click(refresh_datasets)
use_as_reference_button.on_click(use_selected_as_reference)
use_as_comparison_button.on_click(use_selected_as_comparison)
preview_button.on_click(preview_diff)
command_button.on_click(show_command)
start_live_button.on_click(start_live_viewer)
stop_live_button.on_click(stop_live_viewer)

path_controls = widgets.VBox([
    reference_path_input,
    comparison_path_input,
    dataset_path,
])
run_controls = widgets.HBox([
    projection_index,
    position_mode,
    poll_interval,
])
browse_controls = widgets.VBox([
    collection_dir,
    dataset_picker,
    widgets.HBox([refresh_button, use_as_reference_button, use_as_comparison_button]),
])
action_controls = widgets.HBox([
    preview_button,
    command_button,
    start_live_button,
    stop_live_button,
])

help_text = widgets.HTML(
    '<b>Fast path:</b> paste your reference dataset path above, optionally paste a comparison path, then click <b>Preview diff</b> or <b>Start live viewer</b>.'
)

display(help_text, path_controls, run_controls, browse_controls, action_controls, output, live_output)
refresh_datasets()

In [ ]:
gallery_indices = widgets.Text(
    value='0, 100, 237, 500',
    description='Gallery:',
    layout=widgets.Layout(width='500px'),
)
gallery_button = widgets.Button(description='Preview gallery', button_style='success')
gallery_output = widgets.Output()


def parse_gallery_indices(raw: str):
    indices = []
    for part in raw.split(','):
        part = part.strip()
        if not part:
            continue
        indices.append(int(part))
    if not indices:
        raise ValueError('Enter at least one projection index.')
    return indices


def preview_gallery(*_):
    with gallery_output:
        clear_output(wait=True)
        try:
            reference_path, comparison_path, explicit_dataset_path = selected_paths()
            reference_root, reference_scan = ltd.resolve_input_target(reference_path)
            comparison_root, comparison_scan, auto_follow = ltd.resolve_second_target(
                reference_root,
                comparison_path,
                position_mode.value,
            )
            indices = parse_gallery_indices(gallery_indices.value)
            cols = min(2, len(indices))
            rows = (len(indices) + cols - 1) // cols
            fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
            axes = np.atleast_1d(axes).ravel()

            print('Reference dataset:', reference_root)
            print('Comparison dataset:', comparison_root)
            print('Auto follow:', auto_follow)
            print('Gallery indices:', indices)

            for ax, idx in zip(axes, indices):
                ref_image = ltd.load_projection_radiogram(reference_scan, idx, explicit_dataset_path)
                cmp_image = ltd.load_projection_radiogram(comparison_scan, idx, explicit_dataset_path)
                diff = cmp_image - ref_image
                image = ax.imshow(diff, cmap='gray')
                ax.set_title(f'Projection {idx}')
                ax.set_xlabel('X')
                ax.set_ylabel('Y')
                plt.colorbar(image, ax=ax, fraction=0.046, pad=0.04)

            for ax in axes[len(indices):]:
                ax.axis('off')

            fig.suptitle(f'{comparison_root.name} - {reference_root.name}')
            fig.tight_layout()
            plt.show()
        except Exception as exc:
            print(f'Gallery preview failed: {exc}')


gallery_button.on_click(preview_gallery)
display(widgets.HBox([gallery_indices, gallery_button]), gallery_output)

## Notes

- The easiest workflow is to paste a full reference dataset path into `Reference path`.
- `Comparison path` is optional. Leave it empty to auto-follow the newest matching dataset.
- `Start live viewer` updates inline in the notebook instead of launching a separate Python process.
- The browse section is only a helper. You do not need it if you already know the paths.
- `Projection` is zero-based across the whole projection scan, not per `*.h5` block.
- `Positions = same` keeps auto-follow on the same position label as the reference dataset, such as `first_position`.
- `Positions = all` allows auto-follow across all positions in the collection.